<a href="https://colab.research.google.com/github/lygitdata/GarmentIQ/blob/main/test/tutorial_tailor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tutorial - GarmentIQ Tailor

The tailor agent runs the whole GarmentIQ pipeline end to end: classification,
segmentation, optional matting, landmark detection, refinement, derivation, and
measurement. It processes a folder of images and writes masks, annotated images, and
measurement files into an output directory.

This tutorial shows how to configure a tailor agent, how to read the metadata table it
returns, how to swap the segmentation backend, and how to enable alpha matting for cleaner
cutouts.

## Table of Contents

1. [Prerequisites](#prerequisites)
2. [Measure with BiRefNet](#birefnet)
3. [Read the results](#results)
4. [Measure with SAM](#sam)
5. [Add alpha matting](#matting)

<a name="prerequisites"></a>
## Prerequisites

Install the package and download three test images and every model in the pipeline. On
Colab you can keep this section collapsed.

In [ ]:
# @title Install GarmentIQ
!pip install garmentiq -q

In [ ]:
# @title Import GarmentIQ and choose a device

import json

import torch

import garmentiq as giq
from garmentiq.classification.model_definition import tinyViT
from garmentiq.landmark.detection.model_definition import PoseHighResolutionNet
from garmentiq.garment_classes import garment_classes
from garmentiq.landmark.derivation.derivation_dict import derivation_dict
from garmentiq.segmentation.model_definition.birefnet import (
    BiRefNet,
    load_birefnet_config,
)
from garmentiq.segmentation.model_definition.sam import (
    SamModel,
    load_sam_config,
    load_sam_processor,
)
from garmentiq.matting.model_definition.vitmatte import (
    VitMatteForImageMatting,
    load_vitmatte_config,
    load_vitmatte_processor,
)

# GarmentIQ never grabs an accelerator on its own: every model loader and every
# inference function takes a `device` argument that defaults to "cpu".
# The matting stage below runs at full image resolution, and Apple Silicon ("mps")
# can run out of GPU memory and return a subtly wrong matte there, so this tutorial
# prefers CUDA or CPU. Without `do_matte=True`, "mps" is fine.
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

In [ ]:
# @title Download the test images and every model

# cloth_2 is a short sleeve top, cloth_3 a vest dress, cloth_4 a skirt
!mkdir -p ./test_image
!wget -q -O ./test_image/cloth_2.jpg \
    https://raw.githubusercontent.com/lygitdata/GarmentIQ/refs/heads/gh-pages/asset/img/cloth_2.jpg
!wget -q -O ./test_image/cloth_3.jpg \
    https://raw.githubusercontent.com/lygitdata/GarmentIQ/refs/heads/gh-pages/asset/img/cloth_3.jpg
!wget -q -O ./test_image/cloth_4.jpg \
    https://raw.githubusercontent.com/lygitdata/GarmentIQ/refs/heads/gh-pages/asset/img/cloth_4.jpg

!mkdir -p ./models

# Classification
!wget -q -O ./models/tiny_vit_inditex_finetuned.pt \
    https://huggingface.co/lygitdata/garmentiq/resolve/main/tiny_vit_inditex_finetuned.pt

# Landmark detection
!wget -q -O ./models/hrnet.pth \
    https://huggingface.co/lygitdata/garmentiq/resolve/main/hrnet.pth

# Segmentation, BiRefNet
!mkdir -p ./models/birefnet
!wget -q -O ./models/birefnet/model.safetensors \
    https://huggingface.co/lygitdata/BiRefNet_garmentiq_backup/resolve/main/model.safetensors

# Segmentation, SAM 1 base
!mkdir -p ./models/sam_b
!wget -q -O ./models/sam_b/model.safetensors \
    https://huggingface.co/facebook/sam-vit-base/resolve/main/model.safetensors

# Matting, ViTMatte small
!mkdir -p ./models/vitmatte
!wget -q -O ./models/vitmatte/model.safetensors \
    https://huggingface.co/hustvl/vitmatte-small-composition-1k/resolve/main/model.safetensors

print("Downloads finished.")

<a name="birefnet"></a>
## Measure with BiRefNet

A tailor agent is configured once and then reused. `model_dir` is the shared root, and
each `*_model_path` is given relative to it.

`summary()` prints the configuration and the steps that will run, which is worth checking
before a long batch.

In [ ]:
CLASSIFICATION_ARGS = {
    "num_classes": len(garment_classes),
    "img_size": (120, 184),
    "patch_size": 6,
    "resize_dim": (120, 184),
    "normalize_mean": [0.8047, 0.7808, 0.7769],
    "normalize_std": [0.2957, 0.3077, 0.3081],
}

LANDMARK_ARGS = {
    "scale_std": 200.0,
    "resize_dim": [288, 384],
    "normalize_mean": [0.485, 0.456, 0.406],
    "normalize_std": [0.229, 0.224, 0.225],
}

In [ ]:
tailor_biref = giq.tailor(
    input_dir="./test_image",
    model_dir="./models",
    output_dir="./output_biref",
    class_dict=garment_classes,
    do_refine=True,
    do_derive=True,
    derivation_dict=derivation_dict,
    classification_model_path="tiny_vit_inditex_finetuned.pt",
    classification_model_class=tinyViT,
    classification_model_args=CLASSIFICATION_ARGS,
    segmentation_model_path="birefnet/model.safetensors",
    segmentation_model_class=BiRefNet,
    segmentation_model_args={
        "model_config": load_birefnet_config(),
        "resize_dim": (1024, 1024),
        "normalize_mean": [0.485, 0.456, 0.406],
        "normalize_std": [0.229, 0.224, 0.225],
        "background_color": [102, 255, 102],
    },
    landmark_detection_model_path="hrnet.pth",
    landmark_detection_model_class=PoseHighResolutionNet(),
    landmark_detection_model_args=LANDMARK_ARGS,
    device=device,
)

tailor_biref.summary()

In [ ]:
metadata, outputs = tailor_biref.measure(
    save_segmentation_image=True,
    save_measurement_image=True,
)

<a name="results"></a>
## Read the results

`measure` returns a metadata table and the raw outputs. The metadata holds the path of
every file written, which makes the results easy to look up.

In [ ]:
print(metadata)

In [ ]:
# The masks
for image in metadata["mask_image"]:
    giq.landmark.plot(image_path=image, figsize=(3, 3))

In [ ]:
# The images with the background replaced
for image in metadata["bg_modified_image"]:
    giq.landmark.plot(image_path=image, figsize=(3, 3))

In [ ]:
# The images annotated with the measured landmarks
for image in metadata["measurement_image"]:
    giq.landmark.plot(image_path=image, figsize=(3, 3))

In [ ]:
# The measurements themselves
for json_path in metadata["measurement_json"]:
    with open(json_path) as fh:
        print(f"{json_path}:")
        print(json.dumps(json.load(fh), indent=4, sort_keys=True))
        print()

<a name="sam"></a>
## Measure with SAM

Swapping the segmentation backend only changes the three `segmentation_*` arguments. SAM
is prompted, so its arguments carry a processor and a prompt that is applied to every
image in the batch.

In [ ]:
tailor_sam = giq.tailor(
    input_dir="./test_image",
    model_dir="./models",
    output_dir="./output_sam",
    class_dict=garment_classes,
    do_refine=False,
    do_derive=True,
    derivation_dict=derivation_dict,
    classification_model_path="tiny_vit_inditex_finetuned.pt",
    classification_model_class=tinyViT,
    classification_model_args=CLASSIFICATION_ARGS,
    segmentation_model_path="sam_b/model.safetensors",
    segmentation_model_class=SamModel,
    segmentation_model_args={
        "model_config": {"config": load_sam_config("sam-vit-b")},
        "processor": load_sam_processor("sam-vit-b"),
        "prompt": {"points": [[[1000, 900]]]},
        "background_color": [102, 255, 102],
    },
    landmark_detection_model_path="hrnet.pth",
    landmark_detection_model_class=PoseHighResolutionNet(),
    landmark_detection_model_args=LANDMARK_ARGS,
    device=device,
)

tailor_sam.summary()

In [ ]:
metadata_sam, outputs_sam = tailor_sam.measure(
    save_segmentation_image=True,
    save_measurement_image=True,
)

for image in metadata_sam["measurement_image"]:
    giq.landmark.plot(image_path=image, figsize=(3, 3))

<a name="matting"></a>
## Add alpha matting

Set `do_matte=True` and supply a matting model to replace the hard segmentation cutout
with a soft alpha matte. The pipeline then produces an alpha matte and an
alpha-composited image alongside the usual outputs, and landmark detection runs on the
composited image.

Matting itself is covered in the
[matting tutorial](https://colab.research.google.com/github/lygitdata/GarmentIQ/blob/main/test/tutorial_matting.ipynb).

In [ ]:
tailor_matte = giq.tailor(
    input_dir="./test_image",
    model_dir="./models",
    output_dir="./output_matte",
    class_dict=garment_classes,
    do_refine=True,
    do_derive=True,
    derivation_dict=derivation_dict,
    classification_model_path="tiny_vit_inditex_finetuned.pt",
    classification_model_class=tinyViT,
    classification_model_args=CLASSIFICATION_ARGS,
    segmentation_model_path="birefnet/model.safetensors",
    segmentation_model_class=BiRefNet,
    segmentation_model_args={
        "model_config": load_birefnet_config(),
        "resize_dim": (1024, 1024),
        "normalize_mean": [0.485, 0.456, 0.406],
        "normalize_std": [0.229, 0.224, 0.225],
        "background_color": [102, 255, 102],
    },
    landmark_detection_model_path="hrnet.pth",
    landmark_detection_model_class=PoseHighResolutionNet(),
    landmark_detection_model_args=LANDMARK_ARGS,
    do_matte=True,
    matting_model_path="vitmatte/model.safetensors",
    matting_model_class=VitMatteForImageMatting,
    matting_model_args={
        "model_config": {
            "config": load_vitmatte_config("vitmatte-small-composition-1k")
        },
        "processor": load_vitmatte_processor("vitmatte-small-composition-1k"),
        "trimap_args": {"erode_size": 15, "dilate_size": 15},
        "background_color": [102, 255, 102],
    },
    device=device,
)

tailor_matte.summary()

In [ ]:
metadata_matte, outputs_matte = tailor_matte.measure(
    save_segmentation_image=True,
    save_measurement_image=True,
    save_matting_image=True,
)

# Two new columns appear in the metadata
print([c for c in metadata_matte.columns if "matte" in c or "matting" in c])

In [ ]:
# The alpha mattes
for image in metadata_matte["matte_image"]:
    giq.landmark.plot(image_path=image, figsize=(3, 3))

# The alpha-composited images
for image in metadata_matte["matte_composite_image"]:
    giq.landmark.plot(image_path=image, figsize=(3, 3))

### How matting interacts with the rest of the pipeline

**Matting requires segmentation.** The segmentation mask is what supplies ViTMatte's
trimap and Matting Anything's guidance, so `do_matte=True` forces the segmentation stage to
run. Standalone matting has no such requirement: call `garmentiq.matting.matte` with
whatever image and trimap you already have.

**Matting also feeds landmark detection.** The pose model works best on a
background-replaced image, so when matting is enabled the alpha composite is what gets
measured:

| `do_matte` | segmentation `background_color` | matting `background_color` | Detection runs on | Composited onto |
|:---:|:---:|:---:|---|---|
| yes | set | set | soft alpha composite | matting color |
| yes | set | — | soft alpha composite | segmentation color |
| yes | — | set | soft alpha composite | matting color |
| yes | — | — | soft alpha composite | white, a neutral default |
| no | set | — | hard background-modified image | segmentation color |
| no | — | — | the original image | – |

**Which outputs you get depends on what you asked for:**

| Output | Requires |
|---|---|
| `matte_image`, the alpha matte itself | `save_matting_image=True`, no color needed |
| `matte_composite_image` | `save_matting_image=True` **and** a matting `background_color` |
| `bg_modified_image` | a segmentation `background_color` |

**The configuration is validated up front**, so a mistake fails immediately rather than
part way through a long batch:

| Situation | Result |
|---|---|
| `do_matte=True` without a matting model | `ValueError` naming the missing argument |
| `do_matte=True` with ViTMatte but no `processor` | `ValueError` |
| `do_matte=True` with Matting Anything but no `prompt` | `ValueError` |
| `save_matting_image=True` without `do_matte=True` | `ValueError` |

In [ ]:
# Asking to save matting output when matting is disabled
try:
    tailor_biref.measure(save_matting_image=True)
except ValueError as e:
    print("Expected error:", e)